# 36장 · 검색 평가 지표와 답변 근거 검증

검색 평가는 기대한 문서가 상위 몇 개에 포함됐는지, 몇 번째에 처음 나왔는지를 봅니다. 답변이 실제 원문에 맞는지는 별도 항목입니다.

고정된 후보 순위로 Recall@k와 첫 적중 순위의 역수를 계산합니다. 검색 적중 여부와 생성 답변의 사실·인용 정확성은 따로 평가합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## Recall@k와 첫 적중 순위의 역수

Recall@k는 기대 원문 중 앞 k개에서 찾은 원문의 비율입니다. 첫 정답의 순위를 보려면 순위의 역수(RR)를 구합니다. 여러 답변 가능 질문의 RR을 평균한 값이 MRR입니다.

**top-k와 첫 적중 순위**　기대 근거가 2위에 하나 있습니다. k = 2에서 Recall은 1, 첫 적중 순위의 역수는 0.5입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter36/index.html#practice-space)


### 셀 01 · Recall@k 계산

대출 질문의 기대 원문은 library-1입니다. 목록 맨 앞에는 휴관 문서가 잘못 올라왔습니다.


In [1]:
ranked = ['library-2', 'library-1', 'library-3']
gold = {'library-1'}
top = ranked[:2]
recall = len(set(top) & gold) / len(gold)
print(recall)

1.0


1.0은 기대 원문 한 개를 두 문서 안에서 찾았다는 뜻입니다. 대출 문서가 첫 번째였다는 뜻은 아닙니다.

이 예에서 정답 문서는 두 후보 중 둘째입니다. 정답 문서가 한 개라면 두 개까지 읽은 Recall@2는 1입니다. 하지만 첫 번째 후보만 화면에 보여 주면 사용자는 근거를 못 봅니다. 순위까지 평가할 때 둘째 적중에 1/2을 주는 이유입니다.


### 복수 정답 문서의 Recall@k

<p>k는 검색 함수에 요청한 최대 결과 수입니다. Recall@k는 그 결과에서 기대 근거를 얼마나 찾았는지 보는 비율입니다. 이 교재에서는 청크를 원문 ID로 바꾼 집합을 사용합니다. 분자는 검색 원문 집합과 기대 원문 집합의 교집합 크기이고, 분모는 기대 원문 집합의 크기입니다.</p>

<p>예를 들어 기대 원문이 library-1과 library-3 두 개인데 검색 결과가 library-1과 library-2라면, 맞게 찾은 원문은 하나라 recall은 1/2=0.5입니다. 검색 결과가 두 개라는 이유로 분모를 2로 정한 것이 아니라, 질문에 필요한 원문이 두 개여서 분모가 2입니다.</p>

<table><caption>손계산 예 · 모두 가상 계산용</caption><thead><tr><th scope="col">기대 원문</th><th scope="col">검색 원문</th><th scope="col">적중/기대</th><th scope="col">Recall</th></tr>
</thead><tbody><tr><td>1,3</td><td>1,2</td><td>1/2</td><td>0.5</td></tr>
<tr><td>1,3</td><td>1,3</td><td>2/2</td><td>1.0</td></tr>
<tr><td>1</td><td>2,3</td><td>0/1</td><td>0.0</td></tr>
<tr><td>없음</td><td>없음</td><td>0/0</td><td>NA로 별도 처리</td></tr>
</tbody></table>

<p>이 값은 답변의 정확성 점수가 아닙니다. 기대 원문을 찾았더라도 그 안에서 필요한 문장이 잘렸거나 답변이 숫자를 바꿀 수 있습니다. 검색 성과를 설명하는 한 항목으로만 사용합니다.</p>


### 같은 원문의 청크를 한 번만 세기

<p>library-1:0과 library-1:1을 모두 찾았어도 기대 문서가 library-1 하나라면 찾은 원문은 한 개입니다. 원문 ID 단위로 집합을 만든 뒤 교집합을 세어야 Recall이 1을 넘지 않습니다.</p>


### 과제 02 · 첫 적중 순위의 역수

위 상위 두 건에서 첫 기대 원문의 순위를 1부터 세고 역수를 출력하세요.

상위 k개에서 적중이 없으면 이 문항의 점수는 0입니다. 첫 적중을 찾는 식에서 이 경우도 처리하세요.


In [2]:
# enumerate(top, start=1)로 순위를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>first_hit = next((rank for rank, doc in enumerate(top, start=1) if doc in gold), None)
print(1 / first_hit if first_hit else 0.0)</code></pre>
<p>실행 결과</p><pre><code>0.5</code></pre>
<p>두 번째에 처음 맞았으므로 0.5입니다. 이 한 문항의 RR을 다른 문항과 합치기 전에 k와 평가 대상을 같게 정하세요.</p>
</details>


### 셀 03 · 같은 k에서 문항별 점수 평균

앞에서 구한 대출 질문의 RR은 0.5입니다. 두 번째 휴관 질문도 상위 두 건에서 첫 적중을 확인한 뒤 두 문항의 평균을 냅니다.


In [3]:
from statistics import mean
holiday_top = ['library-2', 'library-1']
holiday_rank = holiday_top.index('library-2') + 1
print(mean([0.5, 1 / holiday_rank]))

0.75


0.75는 이 두 답변 가능 문항의 MRR@2입니다. 두 번째 문항에서는 첫 번째 순위에 정답이 와서 RR이 1이었습니다. 전체 검색기의 일반 성능은 이 두 문항만으로 알 수 없습니다.


### 적중 실패와 실행 실패를 분리하기

<p>상위 두 건에 정답이 없으면 답변 가능한 문항의 RR@2는 0입니다. 검색 서버가 시간 안에 답하지 못했다면 순위 자체를 얻지 못했습니다. 같은 평균값이라도 몇 문항을 실행했고 몇 건이 실패했는지 함께 남겨야 합니다.</p>


## 인용 유효성과 답변 정확성

주차 질문처럼 기대 근거가 빈 경우 Recall의 분모는 0입니다. 이때 임의로 1점을 주지 않고 응답 유보를 별도로 봅니다. 근거가 있는 질문의 답도 숫자나 단위를 틀릴 수 있습니다.


### 셀 04 · 인용 유효성과 오답 구분

교육용 오류 문장에서 인용 ID가 기대 근거에 있는지 확인하고 원문과 숫자를 읽어 보세요.


In [4]:
record = {'id': 'library-1',
 'text': '도서 대출 기간은 14일입니다.',
 'source': '가상 도서관 이용 안내',
 'page': 1}
answer = '대출 기간은 999일입니다. [library-1]'
print(f"[{record['id']}]" in answer)
print(record['text'])

True
도서 대출 기간은 14일입니다.


True 다음의 원문에는 14일이라고 적혀 있습니다. 이 예제에서는 미리 제공한 문서 ID가 답에 들어 있지만 숫자가 틀렸습니다. 이 셀은 검색기를 호출하지 않습니다. 자동 판정이나 모델 채점이 무엇을 검사했는지 기준을 남기세요.


### 검색 오류와 생성 오류

<p>검색 단계에서는 질문에 필요한 원문이 결과에 포함됐는지를 봅니다. 답변 단계에서는 검색된 근거를 인용하고 숫자와 조건을 맞게 옮겼는지 봅니다. 마지막 답변만 읽으면 검색 실패와 생성 실패가 섞이므로 검색 문서도 함께 저장합니다.</p>

<p>운영 오류는 정상 검색 결과가 아닙니다. timeout이 났다고 빈 목록으로 바꾸면 원문이 없어서 답하지 못한 것처럼 보입니다. 정상적으로 검색했지만 필요한 원문이 없었던 경우와 호출 자체가 실패한 경우를 나누어야 다음 조치를 정할 수 있습니다.</p>

<table><caption>같은 대출 질문의 다른 실패</caption><thead><tr><th scope="col">관측</th><th scope="col">분류</th><th scope="col">다음 확인</th></tr>
</thead><tbody><tr><td>정상 검색이지만 library-1이 없음</td><td>검색 근거 누락</td><td>질의·청크·k와 검색 조건</td></tr>
<tr><td>검색 중 TimeoutError</td><td>검색 운영 오류</td><td>호출 환경·재현 기록</td></tr>
<tr><td>library-1은 검색했지만 [missing] 인용</td><td>인용 오류</td><td>응답의 인용 ID 대조</td></tr>
<tr><td>library-1을 인용하며 30일 답변</td><td>내용 오류</td><td>원문 14일과 답변 숫자</td></tr>
<tr><td>올바른 검색 후 생성기 TimeoutError</td><td>생성 운영 오류</td><td>검색 점수는 유지하고 생성 실패 기록</td></tr>
</tbody></table>


### 검색과 답변이 같은 문서를 보았는지 확인하기

<p>답변을 평가하기 위해 다시 검색하면 그때의 후보 순서가 달라질 수 있습니다. 답변에 사용한 원문과 그 ID를 함께 보관해 두면 999일이라는 숫자가 어디서 틀렸는지 확인할 수 있습니다.</p>


### 과제 05 · 같은 원문의 두 청크가 1·2위를 차지할 때

기대 원문이 library-1과 library-3입니다. 앞의 두 청크에서 원문 ID를 복원해 Recall@2와 RR@2를 각각 계산하세요. 왜 한 값만 만점인지 설명하세요.


In [ ]:
chunk_ids = ['library-1:0', 'library-1:1', 'library-3:0']
expected = {'library-1', 'library-3'}
# 상위 두 청크의 원문 ID를 사용하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>chunk_ids = [&#x27;library-1:0&#x27;, &#x27;library-1:1&#x27;, &#x27;library-3:0&#x27;]
expected = {&#x27;library-1&#x27;, &#x27;library-3&#x27;}
top_ids = [chunk.rsplit(&#x27;:&#x27;, 1)[0] for chunk in chunk_ids[:2]]
recall = len(set(top_ids) &amp; expected) / len(expected)
rank = next((i for i, doc in enumerate(top_ids, 1) if doc in expected), None)
print(recall, 1 / rank if rank else 0.0)</code></pre>
<p>실행 결과</p><pre><code>0.5 1.0</code></pre>
<p>첫 청크가 기대 원문에서 왔으므로 역순위는 1입니다. 두 번째 청크도 같은 원문에서 왔고 library-3은 잘렸습니다. 필요한 두 원문 중 하나만 찾았으므로 Recall은 0.5입니다.</p>
</details>


## 돌아보기

첫 검색 순위를 낮추는 오류와 답변 숫자를 바꾸는 오류를 서로 다른 문제로 기록해 보세요. 어느 쪽을 고쳐야 하는지 지표 하나만으로는 정할 수 없습니다.
